In [1]:
import re
import argparse

# backend 0 default, backend 1 pp, backend 2, 5, 6 DP, backend 3 TP

def custom_print(message, file_path=None, mode='a'):
    print(message)  # Print to terminal
    if file_path:
        with open(file_path, mode) as file:
            file.write(message + '\n')

def backend_to_comm_type(backend, rank_tid=None):
    if backend == 0:
        return "Default"
    elif backend == 1:
        return "PP"
    elif backend in [2, 5, 6]:
        return "DP"
    elif backend == 3:
        return "TP"
    else:
        return "unknown"
    
def is_same_comm_type(backend1, backend2):
    return backend_to_comm_type(backend1) == backend_to_comm_type(backend2)


def parse_communication_pattern(log_file, rank_tid):
    pattern = re.compile(r"Backend (\d+) RANK (\d+) tid: (\d+) .*?Idx: (\d+),")
    communication_pattern = []

    with open(log_file, 'r') as file:
        start_backend = None
        idx_list = None

        end_backend = None

        for line in file:
            match = pattern.search(line)
            if match:
                backend = int(match.group(1))
                tid = int(match.group(3))
                idx = int(match.group(4))

                # remove TP traffic and backend 0 traffic
                if backend_to_comm_type(backend) == "TP" or backend == 0:
                    continue

                if tid == rank_tid:
                    if not is_same_comm_type(backend, start_backend):
                        # transition to a new backend
                        if start_backend is not None:
                            communication_pattern.append((start_backend, idx_list, end_backend))
                        start_backend = backend
                        idx_list = [idx]
                        end_backend = backend
                    else:
                        idx_list.append(idx)
                        end_backend = backend

        # Append the last backend, rank, and idx
        if start_backend is not None:
            communication_pattern.append((start_backend, idx_list, end_backend))

    return communication_pattern

def get_rank_tid(log_file, node_id, num_local_ranks):
    global_rank_base = node_id * num_local_ranks

    pattern = re.compile(r"Backend 0 RANK (\d+) tid: (\d+)")
    rank_tid_map = {}


    with open(log_file, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                global_rank = int(match.group(1))
                tid = int(match.group(2))
                if global_rank not in rank_tid_map:
                    rank_tid_map[global_rank] = tid

    tid_list = []

    for rank in range(global_rank_base, global_rank_base + num_local_ranks):
        if rank in rank_tid_map:
            tid_list.append(rank_tid_map[rank])
        else:
            raise ValueError(f"Rank {rank} not found in log file {log_file}")

    return tid_list

node_list = ["nid001028", "nid001029", "nid001032", "nid001033"]
num_local_ranks = 4

for i, node in enumerate(node_list):
    log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/deepseek-dp-2-pp-2-tp-4-pm/output-no-controller/torchrun_{node}.ans"
    rank_tid_list = get_rank_tid(log_file, i, num_local_ranks)
    print(rank_tid_list)

    for rank, tid in enumerate(rank_tid_list):
        # rank is local rank

        output_file = f"comm_pattern/comm_pattern_{i}_rank{rank}.txt"
        # Clean the output file before writing
        with open(output_file, 'w') as file:
            file.truncate(0)
        custom_print(f"Node: {node}, Rank: {rank}, Tid: {tid}", output_file)
        pattern = parse_communication_pattern(log_file, tid)

        for backend, idx_list, end_backend in pattern:
            custom_print(f"{backend_to_comm_type(backend)} start_backend: {backend} start_idx: {idx_list[0]} end_backend: {end_backend} end_idx: {idx_list[-1]} len: {len(idx_list)}", output_file)


[140265934706496, 140574484145984, 140030093440832, 140123860903744]
Node: nid001028, Rank: 0, Tid: 140265934706496
PP start_backend: 1 start_idx: 0 end_backend: 1 end_idx: 0 len: 2
DP start_backend: 5 start_idx: 0 end_backend: 5 end_idx: 14 len: 15
PP start_backend: 1 start_idx: 2 end_backend: 1 end_idx: 2 len: 2
DP start_backend: 5 start_idx: 15 end_backend: 5 end_idx: 29 len: 15
PP start_backend: 1 start_idx: 4 end_backend: 1 end_idx: 4 len: 1
DP start_backend: 5 start_idx: 30 end_backend: 5 end_idx: 44 len: 15
PP start_backend: 1 start_idx: 5 end_backend: 1 end_idx: 7 len: 3
DP start_backend: 5 start_idx: 60 end_backend: 5 end_idx: 75 len: 16
PP start_backend: 1 start_idx: 8 end_backend: 1 end_idx: 8 len: 1
DP start_backend: 6 start_idx: 0 end_backend: 5 end_idx: 90 len: 19
PP start_backend: 1 start_idx: 9 end_backend: 1 end_idx: 9 len: 1
DP start_backend: 5 start_idx: 91 end_backend: 5 end_idx: 105 len: 15
PP start_backend: 1 start_idx: 10 end_backend: 1 end_idx: 12 len: 3
DP star

In [2]:
import re
import argparse
import datetime
import numpy as np

# backend 0 default, backend 1 pp, backend 2, 5, 6 DP, backend 3 TP

def custom_print(message, file_path=None, mode='a'):
    print(message)  # Print to terminal
    if file_path:
        with open(file_path, mode) as file:
            file.write(message + '\n')

def backend_to_comm_type(backend, rank_tid=None):
    if backend == 0:
        return "Default"
    elif backend == 1:
        return "PP"
    elif backend in [2, 5, 6]:
        return "DP"
    elif backend == 3:
        return "TP"
    else:
        return "unknown"
    
def is_same_comm_type(backend1, backend2):
    return backend_to_comm_type(backend1) == backend_to_comm_type(backend2)

def time_to_ms(t):
    return (
        t.hour * 3600_000 +
        t.minute * 60_000 +
        t.second * 1000 +
        t.microsecond / 1000.0
    )

def parse_communication_pattern(log_file, rank_tid):
    main_pat = re.compile(
        r"Backend (\d+) RANK (\d+) tid: (\d+) .*?Idx: (\d+),"
    )
    time_pat = re.compile(
        r"t:\[(\d+:\d+:\d+\.\d+)\]"
    )

    communication_pattern = []

    start_backend = None
    end_backend = None

    idx_list = None

    start_time = None
    end_time = None

    # Track per-idx timestamps
    idx_start_times = {}
    idx_end_times = {}

    time_gap = {}

    with open(log_file, 'r') as file:
        for line in file:
            match = main_pat.search(line)
            if not match:
                continue

            backend = int(match.group(1))
            tid = int(match.group(3))
            idx = int(match.group(4))

            # filter
            if backend_to_comm_type(backend) == "TP" or backend == 0:
                continue
            if tid != rank_tid:
                continue

            # extract timestamp if present
            tmatch = time_pat.search(line)
            ts = None
            if tmatch:
                ts = datetime.datetime.strptime(tmatch.group(1), "%H:%M:%S.%f")
                ts = time_to_ms(ts)

            # record pre/post timestamps
            if "pre_coll" in line and ts:
                idx_start_times[(backend, idx)] = ts
            elif "post_coll" in line and ts:
                idx_end_times[(backend, idx)] = ts

            # comm-type transition
            if "pre_coll" in line:
                if not is_same_comm_type(backend, start_backend):
                    if start_backend is not None:
                        communication_pattern.append(
                            (start_backend, idx_list, end_backend, start_time, end_time)
                        )
                    
                    start_time = ts
                    if end_time:
                        time_gap[(start_backend, idx_list[0])] = start_time - end_time
                    start_backend = backend
                    idx_list = [idx]
                else:
                    end_backend = backend
                    idx_list.append(idx)

            if "post_coll" in line:
                end_time = ts

        # flush last segment
        if start_backend is not None:
            communication_pattern.append(
                (start_backend, idx_list, end_backend, start_time, end_time)
            )

    return communication_pattern, time_gap

# log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/dp-2-pp-2-tp-4-pm-8b-provision/output-opus-v3-l_0-b24-provision/torchrun_nid001012.ans"

output_file = "comm_pattern_pp_0.txt"
with open(output_file, 'w') as file:
    file.truncate(0)
log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/deepseek-dp-2-pp-2-tp-4-pm/output-no-controller/torchrun_nid001028.ans"

pattern, time_gap_map_pp0 = parse_communication_pattern(log_file, 140114140198464)

for backend, idx_list, end_backend, start_time, end_time in pattern:
    custom_print(f"{backend_to_comm_type(backend)} start_backend: {backend} start_idx: {idx_list[0]} end_backend: {end_backend} end_idx: {idx_list[-1]} len: {len(idx_list)}, start_time: {start_time}, end_time: {end_time}", output_file)

# if time_gap_list:
#     avg_gap = np.mean(time_gap_list)
#     std_gap = np.std(time_gap_list)
#     min_gap = np.min(time_gap_list)
#     max_gap = np.max(time_gap_list)

#     custom_print(f"Average time gap: {avg_gap:.3f} ms", output_file)
#     custom_print(f"Standard deviation: {std_gap:.3f} ms", output_file)
#     custom_print(f"Minimum time gap: {min_gap:.3f} ms", output_file)
#     custom_print(f"Maximum time gap: {max_gap:.3f} ms", output_file)
# else:
#     custom_print("No time gaps to calculate statistics.")


output_file = "comm_pattern_pp_1.txt"
with open(output_file, 'w') as file:
    file.truncate(0)
log_file = f"<path-to-opus>/Opus/torchtitan/opus-test/deepseek-dp-2-pp-2-tp-4-pm/output-no-controller/torchrun_nid001032.ans"

pattern, time_gap_map_pp1 = parse_communication_pattern(log_file, 140583809054272)

for backend, idx_list, end_backend, start_time, end_time in pattern:
    custom_print(f"{backend_to_comm_type(backend)} start_backend: {backend} start_idx: {idx_list[0]} end_backend: {end_backend} end_idx: {idx_list[-1]} len: {len(idx_list)}, start_time: {start_time}, end_time: {end_time}", output_file)

common_gap = []
for (key1, gap1)in time_gap_map_pp0.items():
    for (key2, gap2) in time_gap_map_pp1.items():
        if key1 == key2 and key1[0] == 1:  # Check if backend and idx are the same
            min_gap = min(gap1, gap2)
            custom_print(f"Backend: {key1[0]}, Idx: {key1[1]}, Min Gap: {min_gap:.3f} ms", output_file)
            common_gap.append(min_gap)

if common_gap:
    avg_gap = np.mean(common_gap)
    std_gap = np.std(common_gap)
    min_gap = np.min(common_gap)
    max_gap = np.max(common_gap)

    custom_print(f"Average time gap: {avg_gap:.3f} ms", output_file)
    custom_print(f"Standard deviation: {std_gap:.3f} ms", output_file)
    custom_print(f"Minimum time gap: {min_gap:.3f} ms", output_file)
    custom_print(f"Maximum time gap: {max_gap:.3f} ms", output_file)

PP start_backend: 1 start_idx: 0 end_backend: 1 end_idx: 1 len: 2, start_time: 10315695.04, end_time: 10315697.112
DP start_backend: 5 start_idx: 0 end_backend: 5 end_idx: 14 len: 15, start_time: 10315799.103, end_time: 10320241.773
PP start_backend: 1 start_idx: 2 end_backend: 1 end_idx: 3 len: 2, start_time: 10320274.511, end_time: 10320279.08
DP start_backend: 5 start_idx: 15 end_backend: 5 end_idx: 29 len: 15, start_time: 10320279.788, end_time: 10322365.405
PP start_backend: 1 start_idx: 4 end_backend: 5 end_idx: 4 len: 1, start_time: 10322395.581, end_time: 10322396.475
DP start_backend: 5 start_idx: 30 end_backend: 5 end_idx: 44 len: 15, start_time: 10324820.779, end_time: 10327477.826
PP start_backend: 1 start_idx: 5 end_backend: 1 end_idx: 6 len: 2, start_time: 10327510.937, end_time: 10327514.2
DP start_backend: 5 start_idx: 45 end_backend: 5 end_idx: 59 len: 15, start_time: 10327518.996, end_time: 10332889.15
PP start_backend: 1 start_idx: 7 end_backend: 5 end_idx: 7 len: 1,